# 🔍 VNCCS_PoseStudio 节点诊断

排查 `ERROR: VNCCS_PoseStudio custom node is missing or not loaded in ComfyUI!` 问题。
逐个 Cell 执行，根据输出判断问题所在。

In [ ]:
# ===== Cell 1: 检查 ComfyUI 连通性 & VNCCS 节点注册状态 =====
import json
import urllib.request
import os

COMFY_URL = os.environ.get("COMFYUI_URL", "http://127.0.0.1:8188").rstrip("/")

def http_get_json(path, timeout=6):
    try:
        with urllib.request.urlopen(f"{COMFY_URL}{path}", timeout=timeout) as r:
            return json.loads(r.read())
    except Exception as e:
        return {"_error": f"{type(e).__name__}: {e}"}

# 1a. 检查 ComfyUI 是否在线
stats = http_get_json("/system_stats")
if "_error" in stats:
    print(f"❌ ComfyUI 不可达 ({COMFY_URL})")
    print(f"   错误: {stats['_error']}")
    print(f"   👉 请先确认 ComfyUI 已启动 (或运行 Cell 1.5 的 colab_quickstart.py)")
else:
    sys_info = stats.get("system", {})
    print(f"✅ ComfyUI 在线: v{sys_info.get('comfyui_version', '?')} @ {COMFY_URL}")
    print(f"   Python: {str(sys_info.get('python_version', '?')).split(' ')[0]}")
    devices = stats.get("devices", [])
    if devices:
        d = devices[0]
        vram = d.get("vram_total")
        vram_gb = f"{round(vram / 2**30)} GiB" if isinstance(vram, (int, float)) and vram else "?"
        print(f"   GPU: {d.get('name', '?')}, VRAM: {vram_gb}")

    # 1b. 检查 VNCCS_PoseStudio 节点
    vnccs = http_get_json("/object_info/VNCCS_PoseStudio")
    if "_error" in vnccs:
        print(f"\n❌ VNCCS 节点查询失败: {vnccs['_error']}")
    elif vnccs.get("VNCCS_PoseStudio"):
        print(f"\n✅ VNCCS_PoseStudio 节点已注册")
        inp = vnccs["VNCCS_PoseStudio"].get("input", {})
        print(f"   inputs: {list(inp.get('required', {}).keys())[:8]}")
    else:
        print(f"\n❌ VNCCS_PoseStudio 节点 **未注册** — 这就是报错的根因！")
        print(f"   👉 需要安装 ComfyUI_VNCCS_Utils 自定义节点 (见下方 Cell 4)")

    # 1c. 顺便检查 QwenImage21Cache
    qcache = http_get_json("/object_info/QwenImage21Cache")
    if qcache.get("QwenImage21Cache"):
        print(f"   QwenImage21Cache 节点: ✅ 已注册")
    else:
        print(f"   QwenImage21Cache 节点: ⚠️ 未注册 (脚本会自动移除它，不影响运行)")

    # 1d. 列出所有 VNCCS 相关节点
    all_nodes = http_get_json("/object_info")
    if "_error" not in all_nodes:
        vnccs_nodes = [k for k in all_nodes if k.startswith("VNCCS")]
        print(f"\n📋 已注册节点总数: {len(all_nodes)}")
        print(f"   其中 VNCCS 相关: {vnccs_nodes if vnccs_nodes else '无 ❌'}")

In [ ]:
# ===== Cell 2: 运行 doctor.py 全量自检 =====
!python scripts/doctor.py

In [ ]:
# ===== Cell 3: 检查 ComfyUI custom_nodes 目录 =====
import os
from pathlib import Path

def find_comfy():
    for p in [os.environ.get('COMFY'), '/content/ComfyUI', str(Path.home() / 'ComfyUI'), '/root/ComfyUI']:
        if p and (Path(p) / 'main.py').exists():
            return Path(p)
    return Path('/content/ComfyUI') if Path('/content').exists() else Path.home() / 'ComfyUI'
COMFY = find_comfy()
os.environ['COMFY'] = str(COMFY)
custom_dir = COMFY / "custom_nodes"
vnccs_dir = custom_dir / "ComfyUI_VNCCS_Utils"

print(f"ComfyUI 根目录: {COMFY}")
print(f"  存在: {'✅' if COMFY.exists() else '❌'}")

print(f"\ncustom_nodes 目录: {custom_dir}")
if custom_dir.exists():
    dirs = sorted([d.name for d in custom_dir.iterdir() if d.is_dir() and not d.name.startswith('.')])
    print(f"  已安装的自定义节点 ({len(dirs)} 个):")
    for d in dirs:
        marker = " ⬅️ VNCCS" if "VNCCS" in d.upper() else ""
        print(f"    📦 {d}{marker}")
    if not any("vnccs" in d.lower() for d in dirs):
        print(f"\n  ❌ ComfyUI_VNCCS_Utils 未找到！")
        print(f"     👉 请运行下方 Cell 4 一键安装")
else:
    print(f"  ❌ custom_nodes 目录不存在")

# 检查 VNCCS LoRA
lora_dir = COMFY / "models" / "loras"
vnccs_lora = lora_dir / "VNCCS_QI2_PoseStudioV1.1.safetensors"
print(f"\nVNCCS LoRA: {vnccs_lora.name}")
print(f"  {'✅ 已存在' if vnccs_lora.exists() else '❌ 未找到 — 需要下载 (见 setup.md §2)'}")


In [ ]:
# ===== Cell 4: 一键安装 ComfyUI_VNCCS_Utils 与 LoRA =====
import os, shutil
from pathlib import Path

def find_comfy():
    for p in [os.environ.get('COMFY'), '/content/ComfyUI', str(Path.home() / 'ComfyUI'), '/root/ComfyUI']:
        if p and (Path(p) / 'main.py').exists():
            return Path(p)
    return Path('/content/ComfyUI') if Path('/content').exists() else Path.home() / 'ComfyUI'
COMFY = find_comfy()
os.environ['COMFY'] = str(COMFY)
print(f"Target ComfyUI path: {COMFY}")

# 4a. 确保目录存在
!mkdir -p {COMFY}/custom_nodes {COMFY}/models/loras

# 4b. Clone VNCCS_Utils 节点（如果不存在）
vnccs_node_dir = COMFY / "custom_nodes" / "ComfyUI_VNCCS_Utils"
if not vnccs_node_dir.exists():
    print("正在拉取 ComfyUI_VNCCS_Utils 插件...")
    !git clone https://github.com/AHEKOT/ComfyUI_VNCCS_Utils.git {COMFY}/custom_nodes/ComfyUI_VNCCS_Utils
else:
    print(f"✅ 节点目录已存在: {vnccs_node_dir}")

# 4c. 安装节点 Python 依赖
!pip install pytorch-lightning roma "dynamicprompts>=0.31,<1"

# 4d. 配置 VNCCS LoRA（优先从 /root/ComfyUI 搬移已下载的文件，避免重复下载）
target_lora = COMFY / "models" / "loras" / "VNCCS_QI2_PoseStudioV1.1.safetensors"
root_lora = Path("/root/ComfyUI/models/loras/VNCCS_QI2_PoseStudioV1.1.safetensors")
if not target_lora.exists():
    if root_lora.exists():
        print(f"检测到已在 /root 下载过 LoRA，正在复制到 {target_lora}...")
        shutil.copyfile(root_lora, target_lora)
        print("✅ 复制完成！")
    else:
        print("正在下载 VNCCS LoRA (159MB)...")
        !wget -c -P {COMFY}/models/loras "https://hf-mirror.com/MIUProject/VNCCS_PoseStudio_QI2.1/resolve/main/VNCCS_QI2_PoseStudioV1.1.safetensors"
else:
    print(f"✅ VNCCS LoRA 已就绪: {target_lora}")

print("\n" + "="*60)
print("✅ 安装步骤完成！请紧接着运行【Cell 5】重启 ComfyUI。")
print("="*60)


In [ ]:
# ===== Cell 5: 强制重启 ComfyUI 并验证 VNCCS 节点 =====
# ⚠️ 必须终止旧进程再启动，ComfyUI 才会扫描新插件！
import os, sys, time, subprocess, urllib.request, json
from pathlib import Path

def find_comfy():
    for p in [os.environ.get('COMFY'), '/content/ComfyUI', str(Path.home() / 'ComfyUI'), '/root/ComfyUI']:
        if p and (Path(p) / 'main.py').exists():
            return Path(p)
    return Path('/content/ComfyUI') if Path('/content').exists() else Path.home() / 'ComfyUI'
COMFY = find_comfy()
os.environ['COMFY'] = str(COMFY)
COMFY_URL = os.environ.get("COMFYUI_URL", "http://127.0.0.1:8188").rstrip("/")

# 5a. 检查节点目录是否存在
vnccs_dir = COMFY / "custom_nodes" / "ComfyUI_VNCCS_Utils"
print(f"1. 检查节点目录: {'✅ 已就绪: ' + str(vnccs_dir) if vnccs_dir.exists() else '❌ 不存在！请先跑 Cell 4'}")

# 5b. 终止旧的 ComfyUI 进程
print("\n2. 强制终止运行中的旧 ComfyUI 进程...")
subprocess.run(["pkill", "-9", "-f", "main.py"])
time.sleep(2)

# 5c. 重新启动 ComfyUI 并等待就绪
print("\n3. 启动全新 ComfyUI 进程（自动扫描加载新插件）...")
subprocess.run([sys.executable, "scripts/colab_quickstart.py", "--setup-only", "--skip-doctor"])

# 5d. 验证 VNCCS_PoseStudio 注册状态
print("\n4. 验证 VNCCS_PoseStudio 注册状态...")
try:
    with urllib.request.urlopen(f"{COMFY_URL}/object_info/VNCCS_PoseStudio", timeout=8) as r:
        data = json.loads(r.read())
        if data.get("VNCCS_PoseStudio"):
            print("\n" + "="*60)
            print("🎉🎉🎉 成功！VNCCS_PoseStudio 节点已成功注册并生效！")
            print("👉 现在可以返回 photobook_dual_mode_example.ipynb 运行写真出图了！")
            print("="*60)
        else:
            print("\n❌ 节点仍然未注册。可能插件加载时遇到了错误。")
            print("正在读取 ComfyUI 启动日志 comfy.log 排查原因：\n")
            logfile = COMFY / "comfy.log"
            if logfile.exists():
                lines = logfile.read_text(errors="replace").splitlines()
                print("\n".join(lines[-40:]))
except Exception as e:
    print(f"❌ 无法连接 ComfyUI: {e}")
